# 09 · Set 3b: can chain-of-thought fix the 3B's retrieval? (GPU, ~35–40 min on A100)

The 3B retriever hands over both needed paragraphs for only ~15% of questions (the 7B: ~45%). Here the **3B retrieves
with chain-of-thought**: it first names what the first part of the question needs and which paragraph has it, then uses that
fact to find the second paragraph, and ends with a `Selected references:` line. Only that final line counts as its selection.

- Same 400 questions, same 20 candidate paragraphs, 3 retrieval samples (as before; results for 1 and 3 merged samples).
- **Retrieval quality**: how often both gold paragraphs are found, one, or none; paragraph F1; tokens and compute.
- **End-to-end**: the four answerers (3B, 3B + CoT, 7B, 7B + CoT) answer on the new 3B + CoT selections, compared with the
  same answerers on the plain 3B and 7B selections (from notebook 04b), and on the gold paragraphs if notebook 08 has run.
- `RUN_7B_COT_RETRIEVAL = True` also runs the 7B with CoT retrieval as a reference (~+15 min).

Output: `runs/set3_retrieval_cot/<timestamp>_<GPU>/diagnose/summary.md` (+ per-question CSVs).

**Runtime → Change runtime type → A100 GPU.**

In [1]:
# ==== settings ====
PROJECT_DIR = "/content/drive/MyDrive/0_EngD_assignments/0_Small_Big_Model_Switching"
STEP0_EXPERIMENT = "env_qwen_hotpotqa_cot"   # the 24-setup table from notebook 04b
STEP0_RUN_DIR = ""                           # empty = latest finished run there (20260929-140148_A100); or paste it
EXPERIMENT  = "set3_retrieval_cot"
LARGE_MODEL_DIR = "Qwen2.5-7B-Instruct"
RUN_7B_COT_RETRIEVAL = False   # True adds 7B + CoT retrieval as a reference (~+15 min)
INCLUDE_GOLD_FROM_08 = True    # add notebook 08's gold-paragraph answers to the final table, if that run exists
TIME_LIMIT_MIN = 55          # every stage stops cleanly at this limit and keeps what is done
RESUME_RUN_DIR = None        # an interrupted run folder, to continue it
DRY_RUN = False              # True = fake models on CPU, tests the whole notebook in ~1 min

In [2]:
import subprocess, sys, os, json, shutil, time
gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader 2>/dev/null",
                     shell=True, capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "none")
if not DRY_RUN:
    assert gpu, "No GPU: Runtime -> Change runtime type -> A100 GPU (or set DRY_RUN = True)."
    if "T4" in gpu:
        raise SystemExit("This runtime has a T4, which cannot run this: the 7B model alone (15 GB) does not fit in its memory.\n"
                         "Runtime -> Change runtime type -> A100 GPU (or G4 / L4) -> Save, then run all cells again.")
    if "A100" not in gpu and "G4" not in gpu and "RTX PRO 6000" not in gpu:
        print("Note: planned for an A100 or G4; this GPU works but timings will differ.")
GPU_TAG = "cpu" if not gpu else gpu.split(",")[0].replace("NVIDIA", "").replace("Tesla", "").strip().split()[0].split("-")[0]
T_START = time.time()

GPU: NVIDIA A100-SXM4-40GB, 40960 MiB, 8.0


In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
import glob
if not STEP0_RUN_DIR:   # empty = the latest finished results-table run in runs/<STEP0_EXPERIMENT> (dry runs skipped)
    done = [p for p in sorted(glob.glob(f"{PROJECT_DIR}/runs/{STEP0_EXPERIMENT}/*/table/per_question.csv")) if "_dryrun" not in p or DRY_RUN]
    assert done, f"No finished results table in runs/{STEP0_EXPERIMENT} - run notebook {'04b' if 'cot' in STEP0_EXPERIMENT else '04'} first."
    STEP0_RUN_DIR = os.path.dirname(os.path.dirname(done[-1]))
assert os.path.exists(f"{STEP0_RUN_DIR}/table/per_question.csv"), f"{STEP0_RUN_DIR} is not a finished results-table run."
print("Results table:", STEP0_RUN_DIR)
DATA_DIR = json.load(open(f"{STEP0_RUN_DIR}/config.json"))["data_dir"]

Results table: /content/drive/MyDrive/0_EngD_assignments/0_Small_Big_Model_Switching/runs/env_qwen_hotpotqa_cot/20260929-140148_A100


In [5]:
from datetime import datetime
from zoneinfo import ZoneInfo
if RESUME_RUN_DIR:
    RUN_DIR = RESUME_RUN_DIR
else:
    stamp = datetime.now(ZoneInfo("Asia/Singapore")).strftime("%Y%m%d-%H%M%S")
    RUN_DIR = f"{PROJECT_DIR}/runs/{EXPERIMENT}/{stamp}_{GPU_TAG}" + ("_dryrun" if DRY_RUN else "")
for sub in ["logs", "code"]:
    os.makedirs(f"{RUN_DIR}/{sub}", exist_ok=True)
print("Run folder:", RUN_DIR)
SRC_DIR = STEP0_RUN_DIR   # the 24-setup table from notebook 04b: its retrievals and answers are the baselines here
os.makedirs(f"{RUN_DIR}/step0", exist_ok=True)
os.makedirs(f"{RUN_DIR}/stages", exist_ok=True)
for sub in ["step0", "stages"]:
    for f in os.listdir(f"{SRC_DIR}/{sub}"):
        if not os.path.exists(f"{RUN_DIR}/{sub}/{f}"):
            shutil.copy(f"{SRC_DIR}/{sub}/{f}", f"{RUN_DIR}/{sub}/{f}")
print("copied baseline retrievals, answers and stage timings from", SRC_DIR)
if not os.path.exists(f"{RUN_DIR}/config.json"):
    json.dump({"experiment": EXPERIMENT, "set": "3", "source_table_run": SRC_DIR, "data_dir": DATA_DIR, "gpu": gpu,
               "dry_run": DRY_RUN, "mode": "chain-of-thought retrieval", "run_7b_cot_retrieval": RUN_7B_COT_RETRIEVAL,
               "started": datetime.now(ZoneInfo("Asia/Singapore")).isoformat()}, open(f"{RUN_DIR}/config.json", "w"), indent=1)

Run folder: /content/drive/MyDrive/0_EngD_assignments/0_Small_Big_Model_Switching/runs/set3_retrieval_cot/20260930-224807_A100
copied baseline retrievals, answers and stage timings from /content/drive/MyDrive/0_EngD_assignments/0_Small_Big_Model_Switching/runs/env_qwen_hotpotqa_cot/20260929-140148_A100


In [6]:
BACKEND = "mock" if DRY_RUN else "vllm"
CODE = "/content/ssw.py"
shutil.copy(f"{PROJECT_DIR}/code/ssw.py", CODE)
shutil.copy(CODE, f"{RUN_DIR}/code/ssw.py")          # keep the exact code with the results

def minutes_left():
    return max(1.0, TIME_LIMIT_MIN - (time.time() - T_START) / 60)

def run_stage(args, log_name):
    """Runs one ssw.py command in a fresh process; streams output here and into logs/<log_name>."""
    cmd = [sys.executable, "-u", CODE] + [str(a) for a in args]
    t0 = time.time()
    with open(f"{RUN_DIR}/logs/{log_name}", "a") as log:
        log.write("\n$ " + " ".join(cmd) + "\n")
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            log.write(line)
            if any(k in line for k in ("it/s]", "s/it]", "INFO ", "(EngineCore")):
                continue                      # keep the notebook readable; everything is in the log file
            print(line, end="")
        p.wait()
    TIMES[log_name.replace(".log", "")] = round(time.time() - t0)
    json.dump(TIMES, open(f"{RUN_DIR}/logs/stage_times.json", "w"), indent=1)
    print(f"--- {log_name}: exit {p.returncode}, {time.time() - t0:.0f}s, {minutes_left():.0f} min left of the limit")
    if p.returncode != 0:
        raise RuntimeError(f"stage failed - see {RUN_DIR}/logs/{log_name}")
TIMES = {}

In [7]:
# vLLM runs only inside the stage processes, so no runtime restart is needed (~3-5 min)
if not DRY_RUN:
    !pip install -q vllm
    !pip uninstall -y -q torchaudio   # Colab's torchaudio is built for another CUDA version and breaks imports
    chk = subprocess.run([sys.executable, "-c", "from vllm import LLM; import vllm, torch; print('vLLM', vllm.__version__, '| torch', torch.__version__)"],
                         capture_output=True, text=True)
    print(chk.stdout or chk.stderr[-2000:])
    assert chk.returncode == 0, "vLLM does not import - Runtime -> Disconnect and delete runtime, then run again."
TIMES["install_vllm"] = round(time.time() - T_START)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.9/314.9 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 100.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 99.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 87.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 729.

In [8]:
LOCAL = "/content/models"
MODELS = {
    "S": {"id": "Qwen/Qwen2.5-3B-Instruct", "dir": "Qwen2.5-3B-Instruct", "params": 3.09e9, "nominal": 3e9},
    "L": {"id": "Qwen/Qwen2.5-7B-Instruct", "dir": LARGE_MODEL_DIR,       "params": 7.62e9, "nominal": 7e9},
}
_t = time.time()
for key in ["S", "L"]:
    m = MODELS[key] if key in MODELS else {"dir": key}
    src, dst = f"{PROJECT_DIR}/models/{m['dir']}", f"{LOCAL}/{m['dir']}"
    m["path"] = dst
    if DRY_RUN:
        continue
    assert os.path.exists(f"{src}/config.json"), f"{src} missing - run 00_setup_download_qwen7b first."
    if not os.path.exists(f"{dst}/config.json"):
        print("copying", m["dir"], "to local disk ...")
        shutil.copytree(src, dst, dirs_exist_ok=True)
for m in MODELS.values():
    m["path"] = f"{LOCAL}/{m['dir']}"
json.dump(MODELS, open(f"{RUN_DIR}/models.json", "w"), indent=1)
TIMES["copy_models"] = round(time.time() - _t)
print(f"models ready in {time.time() - _t:.0f}s")

copying Qwen2.5-3B-Instruct to local disk ...
copying Qwen2.5-7B-Instruct to local disk ...
models ready in 601s


In [9]:
COMMON = ["--run_dir", RUN_DIR, "--data_dir", DATA_DIR, "--models_json", f"{RUN_DIR}/models.json", "--backend", BACKEND]

### Retrieval with chain-of-thought (3B; optionally 7B)

In [10]:
RET_TAGS = ["Sc"] + (["Lc"] if RUN_7B_COT_RETRIEVAL else [])
for k in ["S"] + (["L"] if RUN_7B_COT_RETRIEVAL else []):
    run_stage(["retrieve", "--cot", "--model_key", k, "--time_limit_min", minutes_left()] + COMMON, f"01_retrieve_{k}c.log")

[15:02:05] retrieve (chain-of-thought) | Qwen/Qwen2.5-3B-Instruct | 400 questions (0 already done) | 3 samples
[15:06:02]   20/400
[15:06:09]   40/400
[15:06:14]   60/400
[15:06:20]   80/400
[15:06:26]   100/400
[15:06:31]   120/400
[15:06:37]   140/400
[15:06:42]   160/400
[15:06:47]   180/400
[15:06:52]   200/400
[15:06:58]   220/400
[15:07:03]   240/400
[15:07:08]   260/400
[15:07:15]   280/400
[15:07:20]   300/400
[15:07:26]   320/400
[15:07:32]   340/400
[15:07:36]   360/400
[15:07:42]   380/400
[15:07:47]   400/400
[15:07:47] retrieve done: 127s generation + 215s load
--- 01_retrieve_Sc.log: exit 0, 350s, 35 min left of the limit


### Answer on the new selections: 3B then 7B, each without and with CoT

In [11]:
for k in ["S", "L"]:
    run_stage(["answer", "--model_key", k, "--ret_tags", *RET_TAGS, "--ret_ks", 1, "--out_suffix", "__retc",
               "--time_limit_min", minutes_left()] + COMMON, f"02_answer_{k}.log")
    run_stage(["answer", "--cot", "--model_key", k, "--ret_tags", *RET_TAGS, "--ret_ks", 1, "--out_suffix", "__retc",
               "--time_limit_min", minutes_left()] + COMMON, f"02_answer_{k}c.log")

[15:07:51] answer | Qwen/Qwen2.5-3B-Instruct | retrievers ['Sc'] | 400 questions (0 done) | 5 answer samples + fusion
[15:08:40]   20/400 | answer prompts 20, fuser prompts 40
[15:08:43]   40/400 | answer prompts 20, fuser prompts 40
[15:08:46]   60/400 | answer prompts 20, fuser prompts 40
[15:08:49]   80/400 | answer prompts 20, fuser prompts 40
[15:08:52]   100/400 | answer prompts 20, fuser prompts 40
[15:08:55]   120/400 | answer prompts 20, fuser prompts 40
[15:08:58]   140/400 | answer prompts 20, fuser prompts 40
[15:09:01]   160/400 | answer prompts 20, fuser prompts 40
[15:09:06]   180/400 | answer prompts 20, fuser prompts 40
[15:09:09]   200/400 | answer prompts 20, fuser prompts 40
[15:09:13]   220/400 | answer prompts 20, fuser prompts 40
[15:09:16]   240/400 | answer prompts 20, fuser prompts 40
[15:09:19]   260/400 | answer prompts 20, fuser prompts 40
[15:09:23]   280/400 | answer prompts 20, fuser prompts 40
[15:09:26]   300/400 | answer prompts 20, fuser prompts 40
[

In [12]:
if INCLUDE_GOLD_FROM_08:   # latest notebook-08 run built on the same table
    import glob as _g
    runs08 = [d for d in sorted(_g.glob(f"{PROJECT_DIR}/runs/set3_gold_answering/*"))
              if ("_dryrun" not in d or DRY_RUN) and os.path.exists(f"{d}/diagnose/summary.md")
              and json.load(open(f"{d}/config.json")).get("source_table_run") == SRC_DIR]
    if runs08:
        for f in os.listdir(f"{runs08[-1]}/step0"):
            if f.startswith("answers_") and f.endswith("__gold.jsonl"):
                shutil.copy(f"{runs08[-1]}/step0/{f}", f"{RUN_DIR}/step0/{f}")
        print("added gold-paragraph answers from", runs08[-1])
    else:
        print("no finished notebook-08 run yet - the gold row is left out")

added gold-paragraph answers from /content/drive/MyDrive/0_EngD_assignments/0_Small_Big_Model_Switching/runs/set3_gold_answering/20260930-125338_A100


### Tables: retrieval quality, accuracy by retrieval source x answerer, paired differences

In [13]:
run_stage(["diagnose", "--run_dir", RUN_DIR, "--data_dir", DATA_DIR, "--models_json", f"{RUN_DIR}/models.json"], "99_diagnose.log")

# Set 3 diagnostics - 20260930-224807_A100

Questions: 400 (dev + test pooled unless marked). Retrieval uses 1 sample unless marked. Gold = the two paragraphs HotpotQA marks as needed, given directly to the answerer.


## A. Retrieval quality (did the retriever hand over both gold paragraphs?)

| retriever | samples merged | both gold found | one found | none found | paragraph F1 | paragraphs picked | output tokens/sample | TFLOPs/q | final line present |
|---|---|---|---|---|---|---|---|---|---|
| 3B | 1 | 15.0% | 63.7% | 21.2% | 0.509 | 1.7 | 105 | 17.6 | 100% |
| 3B | 3 | 13.0% | 66.5% | 20.5% | 0.507 | 1.7 | 106 | 18.9 | 100% |
| 3B + CoT | 1 | 24.0% | 59.2% | 16.8% | 0.582 | 1.7 | 179 | 18.3 | 98% |
| 3B + CoT | 3 | 28.7% | 57.5% | 13.8% | 0.620 | 1.7 | 182 | 20.6 | 99% |
| 7B | 1 | 45.0% | 51.0% | 4.0% | 0.722 | 1.9 | 195 | 44.8 | 100% |
| 7B | 3 | 45.2% | 51.7% | 3.0% | 0.725 | 1.9 | 193 | 50.7 | 100% |

## B. Answer accuracy (EM) by retrieval source and answerer - dev + test, 4

In [14]:
from IPython.display import Image, Markdown, display
for f in ["diagnose/summary.md"]:
    p = f"{RUN_DIR}/{f}"
    if not os.path.exists(p):
        continue
    display(Markdown(open(p).read()) if p.endswith(".md") else Image(p))
TIMES["total_notebook"] = round(time.time() - T_START)
json.dump(TIMES, open(f"{RUN_DIR}/logs/stage_times.json", "w"), indent=1)
print("Wall-clock (s):", TIMES)

# Set 3 diagnostics - 20260930-224807_A100

Questions: 400 (dev + test pooled unless marked). Retrieval uses 1 sample unless marked. Gold = the two paragraphs HotpotQA marks as needed, given directly to the answerer.


## A. Retrieval quality (did the retriever hand over both gold paragraphs?)

| retriever | samples merged | both gold found | one found | none found | paragraph F1 | paragraphs picked | output tokens/sample | TFLOPs/q | final line present |
|---|---|---|---|---|---|---|---|---|---|
| 3B | 1 | 15.0% | 63.7% | 21.2% | 0.509 | 1.7 | 105 | 17.6 | 100% |
| 3B | 3 | 13.0% | 66.5% | 20.5% | 0.507 | 1.7 | 106 | 18.9 | 100% |
| 3B + CoT | 1 | 24.0% | 59.2% | 16.8% | 0.582 | 1.7 | 179 | 18.3 | 98% |
| 3B + CoT | 3 | 28.7% | 57.5% | 13.8% | 0.620 | 1.7 | 182 | 20.6 | 99% |
| 7B | 1 | 45.0% | 51.0% | 4.0% | 0.722 | 1.9 | 195 | 44.8 | 100% |
| 7B | 3 | 45.2% | 51.7% | 3.0% | 0.725 | 1.9 | 193 | 50.7 | 100% |

## B. Answer accuracy (EM) by retrieval source and answerer - dev + test, 400 questions

| retrieval source | 3B x1 | 3B x5 | 3B + CoT x1 | 3B + CoT x5 | 7B x1 | 7B x5 | 7B + CoT x1 | 7B + CoT x5 |
|---|---|---|---|---|---|---|---|---|
| 3B | 23.0% | 27.5% | 34.2% | 35.8% | 36.5% | 39.5% | 38.2% | 42.8% |
| 3B + CoT | 25.2% | 31.5% | 40.5% | 38.5% | 41.0% | 45.5% | 44.0% | 45.0% |
| 7B | 31.0% | 38.5% | 49.0% | 48.8% | 43.0% | 50.0% | 50.0% | 52.2% |
| gold paragraphs (no retrieval) | 33.5% | 49.0% | 58.5% | 59.8% | 52.0% | 61.0% | 61.0% | 63.2% |

## B. Answer accuracy (EM) by retrieval source and answerer - test only, 200 questions

| retrieval source | 3B x1 | 3B x5 | 3B + CoT x1 | 3B + CoT x5 | 7B x1 | 7B x5 | 7B + CoT x1 | 7B + CoT x5 |
|---|---|---|---|---|---|---|---|---|
| 3B | 25.0% | 27.0% | 37.0% | 37.0% | 39.5% | 42.0% | 38.5% | 44.0% |
| 3B + CoT | 25.5% | 29.5% | 39.0% | 40.5% | 42.5% | 45.5% | 43.0% | 42.0% |
| 7B | 34.5% | 40.0% | 50.5% | 52.0% | 47.0% | 54.5% | 51.5% | 55.0% |
| gold paragraphs (no retrieval) | 35.0% | 50.5% | 60.0% | 64.5% | 55.5% | 65.0% | 64.0% | 67.0% |

## C. Paired differences, 5 answers + fusion (EM points, 95% bootstrap interval, dev + test)


**Does CoT retrieval help? (same answerer)**

| comparison (retriever -> answerer) | difference |
|---|---|
| 3B + CoT -> 3B vs 3B -> 3B | +4.0 [+0.0, +8.3] (n=400) |
| 3B + CoT -> 3B + CoT vs 3B -> 3B + CoT | +2.8 [-2.0, +7.5] (n=400) |
| 3B + CoT -> 7B vs 3B -> 7B | +6.0 [+1.5, +10.7] (n=400) |
| 3B + CoT -> 7B + CoT vs 3B -> 7B + CoT | +2.3 [-2.0, +6.8] (n=400) |

**3B + CoT retrieval vs plain 7B retrieval (same answerer)**

| comparison (retriever -> answerer) | difference |
|---|---|
| 3B + CoT -> 3B vs 7B -> 3B | -7.0 [-11.5, -2.3] (n=400) |
| 3B + CoT -> 3B + CoT vs 7B -> 3B + CoT | -10.2 [-15.3, -5.5] (n=400) |
| 3B + CoT -> 7B vs 7B -> 7B | -4.5 [-8.7, +0.0] (n=400) |
| 3B + CoT -> 7B + CoT vs 7B -> 7B + CoT | -7.2 [-11.5, -3.0] (n=400) |

**How much does perfect retrieval add over the 7B retriever? (same answerer)**

| comparison (retriever -> answerer) | difference |
|---|---|
| gold paragraphs (no retrieval) -> 3B vs 7B -> 3B | +10.5 [+6.5, +14.7] (n=400) |
| gold paragraphs (no retrieval) -> 3B + CoT vs 7B -> 3B + CoT | +11.0 [+6.5, +15.5] (n=400) |
| gold paragraphs (no retrieval) -> 7B vs 7B -> 7B | +11.0 [+7.5, +14.8] (n=400) |
| gold paragraphs (no retrieval) -> 7B + CoT vs 7B -> 7B + CoT | +11.0 [+7.0, +15.0] (n=400) |

**Answerer gaps with perfect retrieval (gold paragraphs)**

| comparison (retriever -> answerer) | difference |
|---|---|
| gold paragraphs (no retrieval) -> 7B vs gold paragraphs (no retrieval) -> 3B | +12.0 [+7.5, +16.5] (n=400) |
| gold paragraphs (no retrieval) -> 3B + CoT vs gold paragraphs (no retrieval) -> 3B | +10.8 [+6.5, +14.8] (n=400) |
| gold paragraphs (no retrieval) -> 7B vs gold paragraphs (no retrieval) -> 3B + CoT | +1.2 [-2.7, +5.2] (n=400) |
| gold paragraphs (no retrieval) -> 7B + CoT vs gold paragraphs (no retrieval) -> 7B | +2.2 [-1.0, +5.5] (n=400) |


Wall-clock (s): {'install_vllm': 265, 'copy_models': 601, '01_retrieve_Sc': 350, '02_answer_S': 114, '02_answer_Sc': 199, '02_answer_L': 202, '02_answer_Lc': 220, '99_diagnose': 2, 'total_notebook': 1958}
